# EduBénin — Fine-tuning GLM-4.7
# Exécuter sur Google Colab avec GPU T4

## Ordre d'exécution
Exécute les cellules **dans l'ordre, une par une**.
N'exécute pas la suivante si la précédente affiche une erreur.

## Cellule 1 — Vérification GPU
Vérifie que Colab a bien attribué un GPU T4.
Si tu vois 'CPU' ou une erreur → Menu **Exécution > Modifier le type d'exécution > GPU T4**

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(" Pas de GPU détecté ! Change le type d'exécution vers GPU T4.")

print(" GPU:", torch.cuda.get_device_name(0))
vram = torch.cuda.get_device_properties(0).total_memory
print(f" VRAM disponible: {vram/1e9:.1f} GB")

: 

## Cellule 2 — Installation des dépendances
Installe tous les outils nécessaires au fine-tuning.
Cette cellule peut prendre 3-5 minutes.

In [ ]:
!pip install -q transformers==4.40.0 datasets peft trl \
  accelerate bitsandbytes sentencepiece \
  fastapi uvicorn nest-asyncio pyngrok

print(" Dépendances installées")

## Cellule 3 — Montage Google Drive
Connecte ton Google Drive pour lire le dataset et sauvegarder le modèle.
Une fenêtre de permission va s'ouvrir → accepte.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/EduBenin'
print(f"Drive monté: {DRIVE_PATH}")

## Cellule 4 — Chargement du dataset
Lit le fichier `edubenin_dataset_v2.json` depuis ton Drive.
Ce fichier doit être dans `Mon Drive/EduBenin/dataset/edubenin_dataset_v2.json`

In [ ]:
import json
import os

dataset_path = f'{DRIVE_PATH}/dataset/edubenin_dataset_v2.json'

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"❌ Dataset introuvable : {dataset_path}\nUploade d'abord edubenin_dataset_v2.json dans Mon Drive/EduBenin/dataset/")

with open(dataset_path) as f:
    data = json.load(f)

print(f" Dataset chargé : {len(data)} exemples")
print(f"   Types : { {ex['type'] for ex in data} }")
print(f"   Aperçu premier exemple : {list(data[0].keys())}")

## Cellule 5 — Formatage du dataset pour GLM-4.7
Convertit chaque exemple du format `messages` (GPT-5.5)
vers le format `conversations` attendu par GLM-4.7,
en ajoutant le system prompt EduBot en tête de chaque conversation.

In [ ]:
EDUBOT_SYSTEM_PROMPT = (
    "Tu es EduBot, tuteur IA du système éducatif secondaire béninois. "
    "Tu réponds toujours en français avec des exemples béninois "
    "(Cotonou, Dantokpa, Porto-Novo, FCFA, noms locaux). "
    "Tu guides l'élève par des questions progressives "
    "sans jamais donner la réponse directement. "
    "Tu encourages l'élève à chaque étape."
)

def formater_glm(exemple):
    conversations = [
        {"role": "system", "content": EDUBOT_SYSTEM_PROMPT}
    ]
    for message in exemple["messages"]:
        conversations.append({
            "role": message["role"],
            "content": message["content"]
        })
    return {"conversations": conversations}

dataset_glm = [formater_glm(ex) for ex in data]

print(f" Dataset formaté : {len(dataset_glm)} exemples")
print("\nAperçu du premier exemple :")
for msg in dataset_glm[0]["conversations"]:
    print(f"  [{msg['role']}]: {msg['content'][:80]}...")

## Cellule 6 — Chargement du modèle GLM-4.7
Télécharge GLM-4.7 depuis HuggingFace et le charge en 4-bit
pour tenir dans les 15 GB de VRAM du T4.
**Cette cellule peut prendre 10-15 minutes.**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "THUDM/glm-4-9b-chat"

print("Chargement tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print("Chargement modèle en 4-bit (peut prendre 10-15 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    load_in_4bit=True
)

print(" Modèle GLM-4.7 chargé !")

## Cellule 7 — Configuration LoRA
Applique LoRA sur le modèle : seuls ~1% des paramètres seront entraînés.
C'est ce qui permet de fine-tuner un modèle de 9 milliards de paramètres
avec seulement 15 GB de VRAM.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA configuré")

## Cellule 8 — Fine-tuning SFT
Lance l'entraînement sur le dataset EduBénin.
3 epochs, sauvegarde automatique sur Drive toutes les 200 steps.
**Durée estimée : 1h à 3h selon la taille du dataset.**

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

def formater_en_texte(exemple):
    texte = ""
    for msg in exemple["conversations"]:
        if msg["role"] == "system":
            texte += f"[SYS]{msg['content']}[/SYS]\n"
        elif msg["role"] == "user":
            texte += f"[USER]{msg['content']}[/USER]\n"
        elif msg["role"] == "assistant":
            texte += f"[ASST]{msg['content']}[/ASST]\n"
    return {"text": texte}

train_ds = Dataset.from_list(dataset_glm)
train_ds = train_ds.map(formater_en_texte)

output_dir = f'{DRIVE_PATH}/model/edubot_glm47'

args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=2048,
)

print("Début du fine-tuning...")
trainer.train()
print("Fine-tuning terminé !")

## Cellule 9 — Sauvegarde du modèle
Sauvegarde EduBot fine-tuné sur ton Google Drive.

In [ ]:
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f" Modèle EduBot sauvegardé : {output_dir}")

## Cellule 10 — Test rapide
Vérifie qu'EduBot répond correctement après le fine-tuning.

In [ ]:
prompt = """[SYS]Tu es EduBot, tuteur IA du système éducatif secondaire béninois.
Tu réponds toujours en français avec des exemples béninois.[/SYS]
[USER]Explique la loi d'Ohm à un élève de 3ème au Bénin.[/USER]
[ASST]"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    output[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Réponse EduBot :")
print(response)

## Cellule 11 — Serveur FastAPI + ngrok
Lance EduBot comme API REST et l'expose sur internet via ngrok.
L'URL affichée est à copier dans ton `.env` local : `COLAB_NGROK_URL=https://xxxx.ngrok.io`

**Prérequis** : avoir ajouté `NGROK_TOKEN` dans les Secrets Colab (icône 🔑 barre latérale).

In [ ]:
import nest_asyncio
import uvicorn
import threading
import torch
from pyngrok import ngrok
from fastapi import FastAPI
from pydantic import BaseModel
from google.colab import userdata

# ── 1. Token ngrok depuis les Secrets Colab ──────────────────
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# ── 2. Serveur FastAPI EduBot ────────────────────────────────
app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    classe: str = ""
    matiere: str = ""

@app.get("/")
def root():
    return {"status": "EduBot GLM-4.7 opérationnel"}

@app.post("/ia/chat")
def chat(req: ChatRequest):
    prompt = (
        f"[SYS]Tu es EduBot, tuteur IA du système éducatif secondaire béninois. "
        f"Tu réponds toujours en français avec des exemples béninois. "
        f"Tu guides l'élève par des questions progressives "
        f"sans jamais donner la réponse directement.[/SYS]\n"
        f"[USER]{req.message}[/USER]\n"
        f"[ASST]"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return {"response": response}

# ── 3. Lancement serveur + tunnel ngrok ──────────────────────
nest_asyncio.apply()

public_url = ngrok.connect(8000)
print(f"URL publique EduBot : {public_url}")
print(f"")
print(f"Copie cette ligne dans ton .env local (edubenin-ia) :")
print(f"   COLAB_NGROK_URL={public_url}")
print(f"")
print(f"  Puis change MODEL_MODE=colab dans le même .env")

threading.Thread(
    target=uvicorn.run,
    args=(app,),
    kwargs={"host": "0.0.0.0", "port": 8000},
    daemon=True
).start()

print(" Serveur EduBot en ligne !")